# 🧠 Replicación de Sparse Autoencoders (Anthropic) con GPT-2

**Mini-Beca · Mechanistic Interpretability**

Este notebook replica la metodología de los papers de Anthropic:
1. **Toy Models of Superposition** (2022) — los modelos representan más features que dimensiones (*superposition*)
2. **Towards Monosemanticity** (2023) — usar *sparse autoencoders* (SAEs) para descomponer las activaciones en features interpretables
3. **Scaling Monosemanticity** (2024) — escalar los SAEs a modelos grandes

Aquí aplicamos la misma técnica a **GPT-2 small**: entrenamos un sparse autoencoder sobre las activaciones del MLP y extraemos features monosemánticas interpretables.

> ⚡ **Ejecuta este notebook en Colab con GPU**: `Runtime → Change runtime type → T4 GPU`

## 📋 Plan del notebook

| Paso | Qué hacemos |
|------|-------------|
| 1 | Instalar dependencias (torch, transformers, datasets) |
| 2 | Cargar GPT-2 small y un dataset de texto |
| 3 | Recolectar activaciones del MLP (capa intermedia) |
| 4 | Definir y entrenar un **Sparse Autoencoder** |
| 5 | Analizar las features extraídas (top activaciones, densidad) |
| 6 | Visualizar y guardar resultados para el video |

**Concepto clave:** los *sparse autoencoders* aprenden un diccionario *sobrecompleto* de features. Cada feature es una dirección en el espacio de activaciones que se activa de forma *escasa* (pocos tokens a la vez). Esto descompone las activaciones polisemánticas en unidades monosemánticas interpretables.

## 1. Instalar dependencias

Ejecuta esta celda para instalar todo lo necesario. En Colab con GPU, torch ya viene con soporte CUDA.

In [ ]:
# Instalar dependencias
!pip install -q transformers datasets accelerate tqdm numpy pandas matplotlib seaborn

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  Sin GPU — activa Runtime → Change runtime type → T4 GPU")

## 2. Configuración

Aquí defines los parámetros clave del experimento. Puedes ajustarlos según el tiempo de cómputo disponible.

In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# ── Configuración del experimento ──────────────────────────────────
CONFIG = {
    # Modelo
    "model_name": "gpt2",               # GPT-2 small (124M params)
    "layer_idx": 6,                     # Capa del transformer a analizar (0-11)
    "hook_point": "mlp",                # 'mlp' = activaciones post-GELU (3072 dims)

    # Dataset
    "dataset": "wikitext",              # wikitext / openwebtext
    "dataset_config": "wikitext-103-raw-v1",
    "max_tokens": 2_000_000,            # Tokens totales para recolectar activaciones
    "seq_len": 128,                     # Longitud de secuencia

    # Sparse Autoencoder
    "dict_size": 4096,                  # Tamaño del diccionario (sobrecompleto vs 3072)
    "l1_coeff": 1e-3,                   # Coeficiente de sparsity (L1)
    "lr": 1e-3,                         # Learning rate (bajo + más pasos = mejores features)
    "batch_size": 4096,                 # Batch size (activaciones por paso)
    "train_steps": 5000,                # Pasos de entrenamiento
    "log_every": 500,                   # Logging cada N pasos

    # Resampling de features muertas (práctica de Anthropic)
    "resample_every": 1000,             # Resamplear features muertas cada N pasos
    "dead_threshold": 0,                # Feature muerta = nunca se activó en el buffer

    # Semilla
    "seed": 42,
}

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Configuración:", json.dumps(CONFIG, indent=2))
print(f"Device: {DEVICE}")

## 3. Cargar GPT-2 y el dataset

Cargamos GPT-2 small y un dataset de texto. Usamos **wikitext-103** (texto limpio de Wikipedia) para recolectar activaciones.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Cargar modelo y tokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["model_name"]).to(DEVICE)
model.eval()

print(f"Modelo: {CONFIG['model_name']}")
print(f"Capas: {model.config.n_layer}, Hidden: {model.config.n_embd}, MLP: {model.config.n_inner}")
print(f"Vocab: {model.config.vocab_size}")

# Verificar la arquitectura del MLP en la capa elegida
# Nota: GPT-2 usa capas Conv1D (lineales con otro nombre), así que
# usamos weight.shape en vez de in_features/out_features.
# Conv1D.weight.shape = (in_features, out_features)
mlp = model.transformer.h[CONFIG["layer_idx"]].mlp
print(f"\nMLP de la capa {CONFIG['layer_idx']}:")
print(f"  c_fc:   {mlp.c_fc.weight.shape[0]} → {mlp.c_fc.weight.shape[1]}")
print(f"  c_proj: {mlp.c_proj.weight.shape[0]} → {mlp.c_proj.weight.shape[1]}")

In [ ]:
from datasets import load_dataset

# Cargar dataset de texto
if CONFIG["dataset"] == "wikitext":
    # Nota: las versiones nuevas de huggingface_hub requieren formato
    # 'namespace/name'. El dataset wikitext vive en 'Salesforce/wikitext'.
    ds = load_dataset("Salesforce/wikitext", CONFIG["dataset_config"], split="train")
    texts = [ex["text"] for ex in ds if ex["text"].strip()]
    print(f"Wikitext: {len(texts)} documentos cargados")
else:
    # openwebtext (streaming, más diverso pero más lento de descargar)
    ds = load_dataset("Skylion007/openwebtext", split="train", streaming=True)
    texts = []
    for i, ex in enumerate(ds):
        if ex["text"].strip():
            texts.append(ex["text"])
        if len(texts) >= 5000:
            break
    print(f"OpenWebText: {len(texts)} documentos cargados")

# Tokenizar en secuencias
def tokenize_batch(texts, seq_len):
    """Tokeniza textos y los agrupa en secuencias de seq_len tokens."""
    all_ids = []
    for t in texts:
        ids = tokenizer.encode(t, add_special_tokens=False)
        all_ids.extend(ids)
    # Recortar a múltiplo de seq_len
    n = (len(all_ids) // seq_len) * seq_len
    all_ids = all_ids[:n]
    return torch.tensor(all_ids).view(-1, seq_len)

sequences = tokenize_batch(texts, CONFIG["seq_len"])
print(f"Secuencias: {sequences.shape} ({sequences.shape[0] * sequences.shape[1]} tokens)")

## 4. Recolectar activaciones del MLP

Pasamos el texto por GPT-2 y guardamos las activaciones intermedias del MLP (post-GELU) en la capa que elegimos. Estas activaciones son las que usaremos para entrenar el SAE.

> **¿Por qué el MLP?** En los papers de Anthropic, las activaciones del MLP son donde se encuentran las features más interpretables. El MLP de GPT-2 tiene una dimensión intermedia de **3072** — ahí es donde la red "procesa" features antes de proyectarlas de vuelta.

In [ ]:
def collect_activations(model, tokenizer, sequences, config):
    """
    Recolecta activaciones del MLP post-GELU en la capa indicada.
    Retorna un tensor de shape (n_activations, mlp_dim).

    Nota: La MLP de GPT-2 es: c_fc → GELU → c_proj → dropout.
    Usamos register_forward_pre_hook en c_proj para capturar la entrada
    a c_proj, que ES la salida post-GELU (la que queremos).
    """
    layer_idx = config["layer_idx"]
    activations = []

    mlp_module = model.transformer.h[layer_idx].mlp
    # Conv1D.weight.shape = (in_features, out_features)
    mlp_dim = mlp_module.c_fc.weight.shape[1]  # 3072

    captured = {}

    # Hook PRE de c_proj: captura la entrada = salida de GELU = post-GELU
    def hook_fn(module, input):
        # input es una tupla, el primer elemento es la activación post-GELU
        captured["acts"] = input[0].detach()

    handle = mlp_module.c_proj.register_forward_pre_hook(hook_fn)

    n_seqs = len(sequences)
    batch_size = 32

    print(f"Recolectando activaciones de la capa {layer_idx} (MLP dim={mlp_dim})...")
    print(f"Procesando {n_seqs} secuencias en batches de {batch_size}...")

    with torch.no_grad():
        for i in tqdm(range(0, n_seqs, batch_size), desc="Activaciones"):
            batch = sequences[i:i+batch_size].to(DEVICE)
            _ = model(batch)
            if "acts" in captured:
                acts = captured["acts"].cpu()
                activations.append(acts.reshape(-1, mlp_dim))

    handle.remove()

    # Concatenar y filtrar ceros (padding)
    all_acts = torch.cat(activations, dim=0)
    normas = torch.norm(all_acts, dim=1)
    mask = normas > 1e-6
    all_acts = all_acts[mask]

    print(f"\nActivaciones recolectadas: {all_acts.shape}")
    print(f"Media de norma: {normas[mask].mean():.4f}")
    print(f"Dimensión del MLP: {mlp_dim}")

    return all_acts

activations = collect_activations(model, tokenizer, sequences, CONFIG)

# Liberar memoria del modelo GPT-2 (ya no lo necesitamos para el entrenamiento)
del model
torch.cuda.empty_cache()
print("\nGPT-2 liberado de GPU. Memoria disponible para entrenar el SAE.")

## 5. Definir el Sparse Autoencoder

La arquitectura sigue fielmente *Towards Monosemanticity*:

$$\mathbf{f}(\mathbf{x}) = \text{ReLU}(\mathbf{W}_e (\mathbf{x} - \mathbf{b}_d) + \mathbf{b}_e)$$
$$\hat{\mathbf{x}} = \mathbf{W}_d \mathbf{f}(\mathbf{x}) + \mathbf{b}_{\text{dec}}$$
$$\mathcal{L} = \|\mathbf{x} - \hat{\mathbf{x}}\|^2 + \lambda \|\mathbf{f}(\mathbf{x})\|_1$$

**Detalles fieles al paper:**
- **Pre-encoder bias** $\mathbf{b}_d$: inicializado a la **mediana geométrica** del dataset (centra los datos)
- **Decoder bias** $\mathbf{b}_{\text{dec}} = -\mathbf{b}_d$ (constreñido)
- **Columnas del decoder normalizadas** (unit norm) — la magnitud de activación de cada feature es la única variable
- **Pesos encoder/decoder NO atados** — el paper muestra que esto da más capacidad representacional

In [ ]:
class SparseAutoencoder(nn.Module):
    """
    Sparse Autoencoder (SAE) — arquitectura fiel a Towards Monosemanticity.

    Encoder:  f(x) = ReLU(W_e (x - b_d) + b_e)
    Decoder:  x_hat = W_d f(x) + b_dec

    Detalles clave del paper:
    - b_d es un "pre-encoder bias" inicializado a la mediana geométrica del dataset
    - b_dec (post-decoder bias) = -b_d (constreñido)
    - Las columnas de W_d están normalizadas (unit norm)
    - Los pesos del encoder y decoder NO están atados (más capacidad)
    """
    def __init__(self, d_model: int, dict_size: int, geometric_median=None):
        super().__init__()
        self.d_model = d_model
        self.dict_size = dict_size

        # Encoder: d_model → dict_size (pesos independientes del decoder)
        self.W_enc = nn.Linear(d_model, dict_size, bias=True)
        # Decoder: dict_size → d_model
        self.W_dec = nn.Linear(dict_size, d_model, bias=True)

        # Pre-encoder bias b_d: inicializado a la mediana geométrica del dataset
        # (aproximada por la mediana por dimensión sobre un submuestreo)
        if geometric_median is not None:
            with torch.no_grad():
                self.pre_encoder_bias = nn.Parameter(geometric_median.clone())
                # b_dec = -b_d (constreñido)
                self.W_dec.bias.data = -geometric_median.clone()
        else:
            self.pre_encoder_bias = nn.Parameter(torch.zeros(d_model))

        # Inicializar encoder como transpuesta del decoder (buen punto de partida)
        with torch.no_grad():
            self.W_enc.weight.data = self.W_dec.weight.data.T.clone()
            self.W_dec.weight.data = F.normalize(self.W_dec.weight.data, dim=0)

    def forward(self, x):
        """
        Args:
            x: (batch, d_model) — activaciones normalizadas del MLP
        Returns:
            x_hat: (batch, d_model) — reconstrucción
            features: (batch, dict_size) — activaciones de features
            l0: float — media de features activas por token
        """
        # Encoder con pre-encoder bias: f(x) = ReLU(W_e (x - b_d) + b_e)
        x_centered = x - self.pre_encoder_bias
        features = F.relu(self.W_enc(x_centered))

        # Decoder con columnas normalizadas
        W_dec_normalized = F.normalize(self.W_dec.weight, dim=0)
        x_hat = features @ W_dec_normalized.T + self.W_dec.bias

        # L0: media de features activas
        l0 = (features > 0).float().sum(dim=1).mean().item()

        return x_hat, features, l0

    def decode(self, features):
        """Decodifica features a espacio de activaciones."""
        W_dec_normalized = F.normalize(self.W_dec.weight, dim=0)
        return features @ W_dec_normalized.T + self.W_dec.bias

    def compute_loss(self, x, l1_coeff):
        """
        Loss = MSE(x, x_hat) + lambda * mean(|features|)
        """
        x_hat, features, l0 = self.forward(x)
        mse_loss = F.mse_loss(x, x_hat)
        l1_loss = features.abs().mean()
        total_loss = mse_loss + l1_coeff * l1_loss
        return total_loss, mse_loss, l1_loss, l0

    def resample_dead_features(self, activations, dead_indices, n_resample=1):
        """
        Resampling de features muertas (práctica estándar de Anthropic).

        Para cada feature muerta, busca activaciones mal reconstruidas y
        re-inicializa la dirección del diccionario con una de ellas.
        """
        device = self.W_dec.weight.device
        with torch.no_grad():
            # Calcular error de reconstrucción para muestras aleatorias
            idx = torch.randint(0, activations.shape[0], (4096,), device=device)
            x = activations[idx]
            x_hat, _, _ = self.forward(x)
            recon_errors = (x - x_hat).norm(dim=1)

            # Las peores reconstrucciones son buenos candidatos para nuevas features
            _, worst_idx = torch.topk(recon_errors, k=min(n_resample * len(dead_indices), 4096))
            candidates = x[worst_idx]

            for i, dead_idx in enumerate(dead_indices):
                if i >= candidates.shape[0]:
                    break
                new_dir = candidates[i]
                new_dir = F.normalize(new_dir, dim=0)

                # Re-inicializar columna del decoder
                self.W_dec.weight.data[:, dead_idx] = new_dir
                # Re-inicializar fila del encoder (transpuesta)
                self.W_enc.weight.data[dead_idx] = new_dir
                # Resetear bias del encoder
                self.W_enc.bias.data[dead_idx] = 0.0

    def get_dead_features(self, feature_activity, threshold=0):
        """
        Devuelve los índices de features que no se han activado en el buffer
        de actividad reciente (feature_activity == threshold).
        """
        return torch.where(feature_activity <= threshold)[0]

# ── Calcular mediana geométrica aproximada (mediana por dimensión) ──
def compute_geometric_median(activations, n_samples=100_000):
    """Aproximación de la mediana geométrica usando la mediana por dimensión."""
    idx = torch.randperm(activations.shape[0])[:n_samples]
    sample = activations[idx]
    return sample.median(dim=0).values

# Verificar arquitectura
d_model = activations.shape[1]  # 3072 (MLP dim)
geo_median = compute_geometric_median(activations)
sae = SparseAutoencoder(d_model, CONFIG["dict_size"], geometric_median=geo_median).to(DEVICE)
print(f"SAE: {d_model} → {CONFIG['dict_size']} → {d_model}")
print(f"Parámetros: {sum(p.numel() for p in sae.parameters()):,}")
print(f"Pre-encoder bias: media={sae.pre_encoder_bias.mean():.4f}, std={sae.pre_encoder_bias.std():.4f}")

# Verificar que el forward funciona
test_x = activations[:4].to(DEVICE)
x_hat, features, l0 = sae(test_x)
print(f"\nTest forward:")
print(f"  Input:  {test_x.shape}")
print(f"  Output: {x_hat.shape}")
print(f"  Features: {features.shape} (L0={l0:.1f}/{CONFIG['dict_size']})")

## 6. Entrenar el Sparse Autoencoder

Entrenamos el SAE para reconstruir las activaciones del MLP con una penalización de sparsity (L1). El objetivo es que cada feature se active de forma **escasa** (pocos tokens) pero que juntas reconstruyan bien las activaciones.

**Prácticas estándar de Anthropic incorporadas:**
1. **Normalización a unit-norm** de las activaciones
2. **Proyección de gradiente** del decoder (interacción Adam ↔ normalización, del paper)
3. **Resampling de features muertas** cada N pasos — re-inicializa features que nunca se activan con activaciones mal reconstruidas
4. **Tracking de features vivas** — métrica clave del paper para saber si el SAE está funcionando

In [ ]:
def train_sae(sae, activations, config, device):
    """
    Entrena el SAE con Adam, siguiendo las prácticas de Towards Monosemanticity:

    1. Normaliza las activaciones a unit-norm
    2. Proyecta el gradiente del decoder fuera de la dirección de los
       vectores del diccionario (interacción Adam ↔ normalización)
    3. Resamplea features muertas periódicamente
    4. Trackea features vivas y densidad
    """
    # Normalizar activaciones a unit-norm
    norms = torch.norm(activations, dim=1, keepdim=True)
    activations_normed = activations / norms.clamp(min=1e-6)

    # Mover a device
    acts = activations_normed.to(device)

    optimizer = torch.optim.Adam(sae.parameters(), lr=config["lr"])
    batch_size = config["batch_size"]
    n = acts.shape[0]

    # Buffer de actividad de features (para detectar features muertas)
    resample_every = config.get("resample_every", 1000)
    dead_threshold = config.get("dead_threshold", 0)
    feature_activity = torch.zeros(sae.dict_size, device=device)

    history = {"loss": [], "mse": [], "l1": [], "l0": [], "alive": []}

    print(f"Entrenando SAE: {config['train_steps']} pasos, batch={batch_size}")
    print(f"Activaciones disponibles: {n:,}")
    print(f"Resampling de features muertas cada {resample_every} pasos")

    pbar = tqdm(range(config["train_steps"]), desc="SAE training")
    for step in pbar:
        # Muestrear batch aleatorio
        idx = torch.randint(0, n, (batch_size,), device=device)
        x = acts[idx]

        # Forward + loss
        total_loss, mse_loss, l1_loss, l0 = sae.compute_loss(x, config["l1_coeff"])

        # Backward
        optimizer.zero_grad()
        total_loss.backward()

        # ── Normalización del decoder con proyección de gradiente ──
        # En vez de re-normalizar tras cada paso (que rompe la coherencia con Adam),
        # proyectamos el gradiente fuera de la dirección de cada vector del diccionario.
        with torch.no_grad():
            W_dec = sae.W_dec.weight.data  # (d_model, dict_size)
            grad = sae.W_dec.weight.grad
            if grad is not None:
                # Proyección: grad -= (grad · W_dec) * W_dec  (por columna)
                proj = (grad * W_dec).sum(dim=0, keepdim=True)  # (1, dict_size)
                grad -= proj * W_dec
                sae.W_dec.weight.grad = grad

        optimizer.step()

        # Re-normalizar columnas del decoder (tras el paso)
        with torch.no_grad():
            sae.W_dec.weight.data = F.normalize(sae.W_dec.weight.data, dim=0)

        # ── Trackear actividad de features ──
        with torch.no_grad():
            _, features_batch, _ = sae(x)
            feature_activity += (features_batch > 0).float().sum(dim=0)

        # ── Resampling de features muertas ──
        if (step + 1) % resample_every == 0:
            # Calcular alive ANTES de zeroar el buffer
            alive = (feature_activity > dead_threshold).sum().item()
            dead = sae.get_dead_features(feature_activity, threshold=dead_threshold)
            if len(dead) > 0:
                sae.resample_dead_features(acts, dead)
                print(f"\n  [step {step+1}] Resampleadas {len(dead)} features muertas")
            # Resetear buffer de actividad
            feature_activity.zero_()
        else:
            alive = (feature_activity > dead_threshold).sum().item()

        # Logging
        history["loss"].append(total_loss.item())
        history["mse"].append(mse_loss.item())
        history["l1"].append(l1_loss.item())
        history["l0"].append(l0)
        history["alive"].append(alive)

        if (step + 1) % config["log_every"] == 0:
            pbar.set_postfix({
                "loss": f"{total_loss.item():.4f}",
                "mse": f"{mse_loss.item():.4f}",
                "l0": f"{l0:.1f}",
                "alive": f"{alive}/{sae.dict_size}",
            })

    return history, activations_normed

history, activations_normed = train_sae(sae, activations, CONFIG, DEVICE)

# Graficar la curva de entrenamiento
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].plot(history["loss"])
axes[0].set_title("Loss total")
axes[0].set_xlabel("Step")
axes[1].plot(history["mse"])
axes[1].set_title("MSE (reconstrucción)")
axes[1].set_xlabel("Step")
axes[2].plot(history["l0"])
axes[2].set_title("L0 (features activas)")
axes[2].set_xlabel("Step")
axes[3].plot(history["alive"])
axes[3].set_title("Features vivas")
axes[3].set_xlabel("Step")
axes[3].axhline(y=CONFIG["dict_size"], color="r", linestyle="--", alpha=0.5, label="Total")
axes[3].legend()
plt.tight_layout()
plt.show()

print(f"\nResultado final:")
print(f"  MSE final: {history['mse'][-1]:.4f}")
print(f"  L0 final: {history['l0'][-1]:.1f} features activas de {CONFIG['dict_size']}")
print(f"  Sparsity: {history['l0'][-1] / CONFIG['dict_size'] * 100:.2f}% de features activas")
print(f"  Features vivas: {history['alive'][-1]} / {CONFIG['dict_size']}")

## 7. Analizar las features extraídas

Ahora viene la parte interesante: **¿qué representan las features del diccionario?**

Para cada feature (columna del diccionario), buscamos los tokens del dataset donde se activa más fuerte. Si la feature es monosemántica, esos tokens deberían compartir un tema común.

También calculamos la **densidad** de cada feature (fracción de tokens donde se activa) — las features interpretables suelen ser escasas pero consistentes.

In [ ]:
def analyze_features(sae, activations_normed, sequences, tokenizer, config, top_k=10):
    """
    Para cada feature, encuentra los tokens donde se activa más fuerte
    y calcula la densidad (fracción de tokens activos).
    """
    sae.eval()
    dict_size = config["dict_size"]
    device = next(sae.parameters()).device

    # Calcular activaciones de features para todos los tokens
    print("Calculando activaciones de features sobre el dataset...")
    all_features = []
    batch_size = 4096
    n = activations_normed.shape[0]

    with torch.no_grad():
        for i in tqdm(range(0, n, batch_size), desc="Features"):
            batch = activations_normed[i:i+batch_size].to(device)
            _, features, _ = sae(batch)
            all_features.append(features.cpu())

    all_features = torch.cat(all_features, dim=0)  # (n_tokens, dict_size)

    # Densidad de cada feature: fracción de tokens donde se activa
    density = (all_features > 0).float().mean(dim=0)  # (dict_size,)

    # Para cada feature, top-k tokens por activación
    top_examples = {}
    for feat_idx in tqdm(range(dict_size), desc="Top activaciones"):
        activations_feat = all_features[:, feat_idx]
        top_idx = torch.topk(activations_feat, k=top_k).indices
        top_examples[feat_idx] = top_idx.tolist()

    return all_features, density, top_examples

# Analizar features
all_features, density, top_examples = analyze_features(
    sae, activations_normed, sequences, tokenizer, CONFIG, top_k=10
)

print(f"\nAnálisis completado:")
print(f"  Features totales: {CONFIG['dict_size']}")
print(f"  Densidad media: {density.mean():.4f}")
print(f"  Features con densidad < 1%: {(density < 0.01).sum()}")
print(f"  Features con densidad < 0.1%: {(density < 0.001).sum()}")

## 8. Visualizar features interpretables

Mostramos los **top tokens** que activan cada feature. Si la feature es monosemántica, verás un patrón claro (ej: tokens de código, tokens de números, tokens de un idioma, etc.).

> 💡 **Tip:** busca features donde los top tokens compartan un tema. Esas son las features monosemánticas que Anthropic describe en sus papers.

In [ ]:
def show_feature_examples(feat_idx, top_examples, sequences, tokenizer, density, n_show=8):
    """
    Muestra los top tokens que activan una feature específica.
    """
    # Convertir índices de activación a índices de token
    # Las activaciones están en orden de las secuencias concatenadas
    seq_len = sequences.shape[1]
    tokens = sequences.reshape(-1)  # todos los tokens en orden

    print(f"=== Feature {feat_idx} (densidad: {density[feat_idx]:.4f}) ===")
    print(f"Top {n_show} tokens que la activan:\n")

    for rank, act_idx in enumerate(top_examples[feat_idx][:n_show]):
        token_id = tokens[act_idx].item()
        token_str = tokenizer.decode([token_id])
        # Contexto: 5 tokens antes y después
        start = max(0, act_idx - 5)
        end = min(len(tokens), act_idx + 6)
        context = tokenizer.decode(tokens[start:end].tolist())
        # Resaltar el token
        context_highlighted = context.replace(token_str, f"**[{token_str}]**", 1)
        print(f"  #{rank+1}: {context_highlighted}")

    print()

# Mostrar algunas features interesantes
# Ordenar features por densidad (las más escasas suelen ser más interpretables)
sorted_feats = torch.argsort(density)

# Mostrar 5 features de baja densidad (potencialmente monosemánticas)
print("=" * 60)
print("FEATURES DE BAJA DENSIDAD (potencialmente monosemánticas)")
print("=" * 60)
for i in range(5):
    feat_idx = sorted_feats[i].item()
    show_feature_examples(feat_idx, top_examples, sequences, tokenizer, density)

# Mostrar 3 features de densidad media
print("=" * 60)
print("FEATURES DE DENSIDAD MEDIA")
print("=" * 60)
mid = len(sorted_feats) // 2
for i in range(3):
    feat_idx = sorted_feats[mid + i].item()
    show_feature_examples(feat_idx, top_examples, sequences, tokenizer, density)

In [ ]:
# Histograma de densidad de features
plt.figure(figsize=(10, 5))
plt.hist(density.numpy(), bins=50, color="#2962FF", alpha=0.8, edgecolor="white")
plt.xlabel("Densidad (fracción de tokens donde se activa)")
plt.ylabel("Número de features")
plt.title("Distribución de densidad de features del SAE")
plt.yscale("log")
plt.grid(alpha=0.3)
plt.show()

# Estadísticas
print(f"Densidad media: {density.mean():.4f}")
print(f"Densidad mediana: {density.median():.4f}")
print(f"Features con densidad < 0.1%: {(density < 0.001).sum()} / {CONFIG['dict_size']}")
print(f"Features con densidad < 1%: {(density < 0.01).sum()} / {CONFIG['dict_size']}")

## 9. Guardar resultados

Guardamos todos los resultados para poder hacer el video explicativo después:
- Pesos del SAE entrenado
- Densidad de features
- Top ejemplos de activación por feature
- Métricas de entrenamiento
- Visualizaciones

> 📥 **Importante:** al final de esta sección descarga el archivo `sae_results.zip` y guárdalo en la carpeta `outputs/` del proyecto local.

In [ ]:
import zipfile
import shutil

# Crear carpeta de resultados
results_dir = "/content/sae_results"
os.makedirs(results_dir, exist_ok=True)

# 1. Guardar pesos del SAE
torch.save(sae.state_dict(), os.path.join(results_dir, "sae_weights.pt"))
print("✓ Pesos del SAE guardados")

# 2. Guardar configuración
with open(os.path.join(results_dir, "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)
print("✓ Configuración guardada")

# 3. Guardar densidad de features
np.save(os.path.join(results_dir, "feature_density.npy"), density.numpy())
print("✓ Densidad de features guardada")

# 4. Guardar top ejemplos (índices de tokens)
with open(os.path.join(results_dir, "top_examples.json"), "w") as f:
    json.dump(top_examples, f)
print("✓ Top ejemplos guardados")

# 5. Guardar métricas de entrenamiento
with open(os.path.join(results_dir, "training_history.json"), "w") as f:
    json.dump({k: v[-100:] for k, v in history.items()}, f)
print("✓ Métricas de entrenamiento guardadas")

# 6. Guardar visualizaciones
plt.figure(figsize=(10, 5))
plt.hist(density.numpy(), bins=50, color="#2962FF", alpha=0.8, edgecolor="white")
plt.xlabel("Densidad")
plt.ylabel("Número de features")
plt.title("Distribución de densidad de features")
plt.yscale("log")
plt.grid(alpha=0.3)
plt.savefig(os.path.join(results_dir, "density_histogram.png"), dpi=150, bbox_inches="tight")
plt.close()
print("✓ Histograma de densidad guardado")

# 7. Guardar un resumen de features interpretables (top 20 por densidad baja)
summary = []
sorted_feats = torch.argsort(density)
for i in range(min(20, len(sorted_feats))):
    feat_idx = sorted_feats[i].item()
    summary.append({
        "feature_idx": feat_idx,
        "density": float(density[feat_idx]),
        "top_tokens": [tokenizer.decode([sequences.reshape(-1)[idx].item()]) for idx in top_examples[feat_idx][:5]],
    })
with open(os.path.join(results_dir, "feature_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("✓ Resumen de features guardado")

# 8. Empaquetar todo en un zip
zip_path = "/content/sae_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(results_dir):
        zf.write(os.path.join(results_dir, fname), arcname=fname)

print(f"\n✅ Resultados empaquetados en: {zip_path}")
print(f"   Tamaño: {os.path.getsize(zip_path) / 1e6:.1f} MB")

# Descargar el zip (en Colab)
try:
    from google.colab import files
    files.download(zip_path)
    print("📥 Descarga iniciada — guarda el archivo en outputs/ del proyecto local")
except ImportError:
    print("No estás en Colab. Descarga manualmente el archivo.")

## ✅ Resumen y próximos pasos

### Qué hemos hecho
1. Cargamos **GPT-2 small** y recolectamos activaciones del MLP (capa 6, dim 3072)
2. Entrenamos un **Sparse Autoencoder** (diccionario de 4096 features) para descomponer esas activaciones
3. Analizamos las features: densidad, top tokens de activación, interpretabilidad

### Qué esperamos observar (comparación con Anthropic)
| Métrica | Anthropic (Claude 3) | Nuestra réplica (GPT-2) |
|---------|---------------------|------------------------|
| Features interpretables | ✅ Muchas | ✅ Esperamos varias |
| Features de baja densidad | ✅ Escasas y específicas | ✅ Deberían aparecer |
| Features de contexto (código, idiomas) | ✅ | ✅ Probable en GPT-2 |
| Features abstractas (personas, lugares) | ✅ | ⚠️ Menos (GPT-2 es pequeño) |

### Próximos pasos
1. **Descarga** `sae_results.zip` → guárdalo en `outputs/` del proyecto local
2. **Analizamos juntos** las features encontradas (¿hay alguna monosemántica clara?)
3. **Hacemos el video** con Manim explicando los resultados reales

> 📌 Si quieres features más abstractas, prueba con:
> - Más tokens de dataset (`max_tokens` = 5M+)
> - Diccionario más grande (`dict_size` = 8192)
> - Más pasos de entrenamiento (`train_steps` = 10000)